# LeetCode 3502 — Minimum Cost to Reach Every Position bằng Python

Link đề: https://leetcode.com/problems/minimum-cost-to-reach-every-position/ · Độ khó: **Easy** · Chủ đề: Array, Prefix Minimum

Skeleton LeetCode:

```python
class Solution:
    def minCosts(self, cost: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bạn đứng **cuối** hàng `n + 1` người; muốn tiến lên **từng vị trí** `i`, mỗi người phía trước bắt phí `cost[i]` để đổi chỗ, còn người phía sau đổi chỗ **miễn phí**; tìm chi phí **nhỏ nhất** để tới **mỗi** vị trí.
2. Rút ra rằng đáp án tại `i` là **giá trị nhỏ nhất trong `cost[0..i]`** (prefix minimum — "min tiền tố").
3. Viết min tiền tố theo 4 cách: brute `O(n^2)`, một biến `cur`, `itertools.accumulate(cost, min)`, và danh sách "kỷ lục" (record lows).
4. Đối chiếu các cách bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: trả lại chính `cost`, dùng min toàn cục, dùng min hậu tố (suffix), và khởi tạo `cur = 0`.

## 2. Đề bài & ví dụ

Bạn đứng ở vị trí `n` (cuối hàng) trong hàng gồm `n + 1` người đánh số `0..n`. Cho mảng `cost` độ dài `n`: `cost[i]` là số tiền người thứ `i` đòi để **đổi chỗ** với bạn. Luật đổi chỗ:

- người **đứng trước** bạn: bạn phải **trả** `cost[i]` để đổi chỗ với họ;
- người **đứng sau** bạn: họ đổi chỗ với bạn **miễn phí**.

Trả về mảng `answer` độ dài `n` mà `answer[i]` là **tổng chi phí nhỏ nhất** để bạn tới vị trí `i`.

| Ví dụ | cost | Output | Giải thích |
|---|---|---|---|
| 1 | `[5,3,4,1,3,2]` | `[5,3,3,1,1,1]` | tới vị trí 0 phải trả 5; tới vị trí 1 trả 3; tới vị trí 2 có thể trả 3 (người 1) rồi đổi miễn phí; tới vị trí 3 trả 1 ... |
| 2 | `[1,2,4,6,7]` | `[1,1,1,1,1]` | người 0 chỉ đòi 1: đổi với họ, rồi mọi vị trí còn lại đổi miễn phí |

Constraints: `n == cost.length`; `1 <= n <= 100`; `1 <= cost[i] <= 100`.

In [1]:
cost = [5, 3, 4, 1, 3, 2]
answer = []
cur = None
for c in cost:
    cur = c if cur is None else min(cur, c)
    answer.append(cur)
print(answer)

[5, 3, 3, 1, 1, 1]


## 3. Kiến thức nền

### 3.1. Prefix minimum (min tiền tố)

Với mảng `a`, **min tiền tố** tại `i` là `min(a[0], a[1], ..., a[i])`. Cả mảng min tiền tố có thể tính trong **một lượt**: `p[0] = a[0]`, `p[i] = min(p[i - 1], a[i])` — mỗi bước chỉ so sánh giá trị hiện có với phần tử mới.

In [2]:
a = [5, 3, 4, 1, 3, 2]
print([min(a[:i + 1]) for i in range(len(a))])
p = [a[0]]
for i in range(1, len(a)):
    p.append(min(p[i - 1], a[i]))
print(p)

[5, 3, 3, 1, 1, 1]
[5, 3, 3, 1, 1, 1]


### 3.2. Cắt lát `a[:i + 1]` và hàm `min`

`a[:i + 1]` lấy các phần tử chỉ số `0..i` (không gồm `i + 1`) — cắt lát tạo **danh sách mới** (`O(i)`). `min(iterable)` duyệt hết để tìm nhỏ nhất. Hai thứ kết hợp cho cách viết brute rất dễ hiểu nhưng `O(n^2)` tổng cộng.

In [3]:
a = [5, 3, 4, 1, 3, 2]
print(a[:1], a[:3], a[:6])
print(min(a[:3]))

[5] [5, 3, 4] [5, 3, 4, 1, 3, 2]
3


### 3.3. `itertools.accumulate(iterable, func)`

`accumulate(a, min)` sinh ra dãy `a[0]`, `min(a[0], a[1])`, `min(min(a[0], a[1]), a[2])`, ... — chính là min tiền tố. Mặc định (không `func`) `accumulate` cho **tổng** tiền tố. Kết quả là một iterator, bọc `list(...)` để xem.

In [4]:
from itertools import accumulate
a = [5, 3, 4, 1, 3, 2]
print(list(accumulate(a)))          # tổng tiền tố
print(list(accumulate(a, min)))     # min tiền tố
print(list(accumulate(a, max)))     # max tiền tố

[5, 8, 12, 13, 16, 18]
[5, 3, 3, 1, 1, 1]
[5, 5, 5, 5, 5, 5]


### 3.4. Giá trị khởi tạo cho biến `cur`

Khi tính min, `cur` phải khởi tạo bằng **phần tử đầu** (hoặc `float('inf')`), **không** phải `0`: vì `min(0, x) = 0` với mọi `x >= 1` nên `cur` mắc kẹt ở `0`. Đây là lỗi kinh điển. Ở bài này `cost[i] >= 1` nên `cur = 0` chắc chắn sai.

In [5]:
cur_bad, cur_ok = 0, float("inf")
for c in [5, 3, 4]:
    cur_bad = min(cur_bad, c)
    cur_ok = min(cur_ok, c)
    print(c, "| cur=0 ->", cur_bad, "| cur=inf ->", cur_ok)

5 | cur=0 -> 0 | cur=inf -> 5
3 | cur=0 -> 0 | cur=inf -> 3
4 | cur=0 -> 0 | cur=inf -> 3


### 3.5. "Kỷ lục" (record low) và `list.extend`

Min tiền tố chỉ **thay đổi** tại các chỉ số mà `cost[i]` **nhỏ hơn mọi giá trị trước đó** (một "kỷ lục mới"). Giữa hai kỷ lục liên tiếp, min tiền tố **không đổi**. Nên ta chỉ cần ghi lại các chỉ số kỷ lục rồi điền đoạn bằng `[giá_trị] * độ_dài_đoạn`.

In [6]:
a = [5, 3, 4, 1, 3, 2]
records = []
best = float("inf")
for i, v in enumerate(a):
    if v < best:
        best = v
        records.append(i)
print("chỉ số kỷ lục:", records, "giá trị:", [a[i] for i in records])

chỉ số kỷ lục: [0, 1, 3] giá trị: [5, 3, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với `cost = [5, 3, 4, 1, 3, 2]` (bạn đứng ở vị trí `6`):

```
vị trí 0: chỉ có cách trả cost[0] = 5                             -> 5
vị trí 1: trả cost[1] = 3 (hoặc trả 5 rồi ... đắt hơn)             -> 3
vị trí 2: trả 3 (người 1) để tới vị trí 1, rồi người 2 phía sau đổi miễn phí -> 3 (rẻ hơn trả cost[2] = 4)
vị trí 3: trả cost[3] = 1                                          -> 1
vị trí 4: người 3 (đã trả 1), rồi đổi miễn phí                      -> 1
vị trí 5: tương tự                                                 -> 1
```

Ý tưởng: để tới vị trí `i`, **trả cho một người `j` nào đó với `j <= i`** (đổi chỗ để tới vị trí `j`), sau đó những người ở phía sau bạn (các vị trí `j+1..i`) **đổi chỗ miễn phí** để bạn "lùi" tới `i`. Vậy chi phí nhỏ nhất = `min(cost[0..i])`.

## 5. Cách 1 — Brute Force / baseline

Với mỗi vị trí `i`, tính `min(cost[:i + 1])` bằng cách duyệt lại từ đầu.

Nút thắt: mỗi `i` duyệt `i + 1` phần tử -> tổng `1 + 2 + ... + n = O(n^2)`. Với `n <= 100` thì vẫn tức thì, nhưng bài chỉ cần **một lượt**, vì `min(cost[:i + 1])` và `min(cost[:i])` khác nhau chỉ ở một phần tử.

In [7]:
class SolutionBrute:
    def minCosts(self, cost):
        n = len(cost)
        ans = []
        for i in range(n):
            best = cost[0]
            for j in range(i + 1):
                if cost[j] < best:
                    best = cost[j]
            ans.append(best)
        return ans

print(SolutionBrute().minCosts([5, 3, 4, 1, 3, 2]))
print(SolutionBrute().minCosts([1, 2, 4, 6, 7]))

[5, 3, 3, 1, 1, 1]
[1, 1, 1, 1, 1]


**Độ phức tạp:** Time `O(n^2)`, Space `O(n)` cho mảng kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** min tiền tố có thể cập nhật **dần**: `best_i = min(best_{i-1}, cost[i])`. Chỉ cần nhớ **một** số — min của các phần tử đã duyệt.

**Quy tắc "nếu… thì…":**

- Nếu `cost[i] < cur` -> `cur = cost[i]` (tìm được người rẻ hơn, từ nay trả cho người này).
- Nếu không -> `cur` giữ nguyên (vẫn tiếp tục dùng người rẻ nhất trước đó, rồi đổi miễn phí).
- Ghi `answer[i] = cur`.

Invariant: sau khi xử lý chỉ số `i`, `cur = min(cost[0..i])` và `answer[0..i]` đã đúng.

In [8]:
cost = [5, 3, 4, 1, 3, 2]
cur = cost[0]
answer = []
for c in cost:
    if c < cur:
        cur = c
    answer.append(cur)
print(answer)

[5, 3, 3, 1, 1, 1]


## 7. Các cách tiếp theo

Ta có 4 cách: (1) brute `min(cost[:i + 1])`; (2) một biến chạy `cur`; (3) `accumulate(cost, min)`; (4) danh sách kỷ lục và điền đoạn.

### Cách 2 — Một biến `cur` (chuẩn, in-place cũng được)

Duyệt một lần, `cur` là min đang chạy. Đây là lời giải tối ưu `O(n)` thời gian, `O(1)` bộ nhớ thêm (ngoài mảng kết quả).

In [9]:
class SolutionRunning:
    def minCosts(self, cost):
        ans = []
        cur = cost[0]
        for c in cost:
            if c < cur:
                cur = c
            ans.append(cur)
        return ans

print(SolutionRunning().minCosts([5, 3, 4, 1, 3, 2]))
print(SolutionRunning().minCosts([1, 2, 4, 6, 7]))

[5, 3, 3, 1, 1, 1]
[1, 1, 1, 1, 1]


Độ phức tạp Cách 2: Time `O(n)`, Space `O(1)` thêm (mảng trả về `O(n)`).

### Cách 3 — `itertools.accumulate(cost, min)`

Thư viện chuẩn có sẵn đúng thao tác "gộp dần bằng `min`".

In [10]:
from itertools import accumulate

class SolutionAccumulate:
    def minCosts(self, cost):
        return list(accumulate(cost, min))

print(SolutionAccumulate().minCosts([5, 3, 4, 1, 3, 2]))
print(SolutionAccumulate().minCosts([1, 2, 4, 6, 7]))

[5, 3, 3, 1, 1, 1]
[1, 1, 1, 1, 1]


Độ phức tạp Cách 3: Time `O(n)`, Space `O(n)` (kết quả).

### Cách 4 — Danh sách kỷ lục và điền đoạn

Tìm các chỉ số `r_0 = 0 < r_1 < ... < r_k` mà `cost[r_t]` là kỷ lục mới. Đoạn `[r_t, r_{t+1})` nhận giá trị `cost[r_t]`; đoạn cuối `[r_k, n)` cũng vậy. Cách này cho thấy **cấu trúc** của đáp án: một dãy **bậc thang giảm dần**.

In [11]:
class SolutionRecords:
    def minCosts(self, cost):
        n = len(cost)
        records = []
        best = float("inf")
        for i, v in enumerate(cost):
            if v < best:
                best = v
                records.append(i)
        ans = []
        for t, r in enumerate(records):
            end = records[t + 1] if t + 1 < len(records) else n
            ans.extend([cost[r]] * (end - r))
        return ans

print(SolutionRecords().minCosts([5, 3, 4, 1, 3, 2]))
print(SolutionRecords().minCosts([1, 2, 4, 6, 7]))

[5, 3, 3, 1, 1, 1]
[1, 1, 1, 1, 1]


Độ phức tạp Cách 4: Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 với `cost = [5, 3, 4, 1, 3, 2]`:

| bước `i` | `cost[i]` | so với `cur` | `cur` sau | `answer` |
|---|---|---|---|---|
| 0 | 5 | `5 < 5`? không | 5 | `[5]` |
| 1 | 3 | `3 < 5` có | 3 | `[5,3]` |
| 2 | 4 | `4 < 3`? không | 3 | `[5,3,3]` |
| 3 | 1 | `1 < 3` có | 1 | `[5,3,3,1]` |
| 4 | 3 | `3 < 1`? không | 1 | `[5,3,3,1,1]` |
| 5 | 2 | `2 < 1`? không | 1 | `[5,3,3,1,1,1]` |

In [12]:
def trace_min_costs(cost):
    cur = cost[0]
    ans = []
    for i, c in enumerate(cost):
        if c < cur:
            note = f"{c} < {cur}: cập nhật cur"
            cur = c
        else:
            note = f"{c} >= {cur}: giữ cur"
        ans.append(cur)
        print(f"i={i}: cost[i]={c} | {note:<24} | cur={cur} | answer={ans}")
    return ans

trace_min_costs([5, 3, 4, 1, 3, 2])
trace_min_costs([1, 2, 4, 6, 7])

i=0: cost[i]=5 | 5 >= 5: giữ cur          | cur=5 | answer=[5]
i=1: cost[i]=3 | 3 < 5: cập nhật cur      | cur=3 | answer=[5, 3]
i=2: cost[i]=4 | 4 >= 3: giữ cur          | cur=3 | answer=[5, 3, 3]
i=3: cost[i]=1 | 1 < 3: cập nhật cur      | cur=1 | answer=[5, 3, 3, 1]
i=4: cost[i]=3 | 3 >= 1: giữ cur          | cur=1 | answer=[5, 3, 3, 1, 1]
i=5: cost[i]=2 | 2 >= 1: giữ cur          | cur=1 | answer=[5, 3, 3, 1, 1, 1]
i=0: cost[i]=1 | 1 >= 1: giữ cur          | cur=1 | answer=[1]
i=1: cost[i]=2 | 2 >= 1: giữ cur          | cur=1 | answer=[1, 1]
i=2: cost[i]=4 | 4 >= 1: giữ cur          | cur=1 | answer=[1, 1, 1]
i=3: cost[i]=6 | 6 >= 1: giữ cur          | cur=1 | answer=[1, 1, 1, 1]
i=4: cost[i]=7 | 7 >= 1: giữ cur          | cur=1 | answer=[1, 1, 1, 1, 1]


[1, 1, 1, 1, 1]

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def minCosts(self, cost: List[int]) -> List[int]:
        ans = []
        cur = cost[0]
        for c in cost:
            if c < cur:
                cur = c
            ans.append(cur)
        return ans

print(Solution().minCosts([5, 3, 4, 1, 3, 2]), Solution().minCosts([1, 2, 4, 6, 7]))

[5, 3, 3, 1, 1, 1] [1, 1, 1, 1, 1]


## 10. Vì sao thuật toán đúng

- **Cận dưới:** để tới vị trí `i` bạn cần ít nhất **một lần trả tiền** cho một người ở vị trí `j <= i` (chỉ những người đứng trước bạn mới đòi tiền; để vượt qua vị trí `i` từ vị trí lớn hơn, bạn phải đi qua một người có chỉ số `<= i` bằng cách trả cho họ). Mỗi lần trả tốn ít nhất `min(cost[0..i])`, nên chi phí `>= min(cost[0..i])`.
- **Đạt được:** gọi `j*` là chỉ số có `cost` nhỏ nhất trong `0..i`. Trả `cost[j*]` để tới vị trí `j*`. Các vị trí `j*+1..i` nằm **phía sau** bạn nên đổi chỗ miễn phí, đưa bạn tới `i`. Tổng chi phí đúng `cost[j*]`.
- **Hai điều trên cho** `answer[i] = min(cost[0..i])`, và công thức cập nhật `best_i = min(best_{i-1}, cost[i])` hiển nhiên đúng.
- **Cách 3, 4:** `accumulate(min)` làm đúng phép cập nhật trên; cách kỷ lục chỉ ghi nhận các bước làm `cur` đổi.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(cost)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute `min(cost[:i + 1])` | `O(n^2)` | `O(n)` |
| 2. Một biến `cur` | `O(n)` | `O(1)` thêm |
| 3. `accumulate(cost, min)` | `O(n)` | `O(n)` |
| 4. Kỷ lục + điền đoạn | `O(n)` | `O(n)` |

Cách 2 là tối ưu về cả hai. Mọi cách đều phải tạo mảng kết quả độ dài `n`.

## 12. Bộ test

Test 1: ví dụ đề và edge case (`n = 1`, dãy tăng, dãy giảm, toàn bằng nhau, nhỏ nhất ở cuối, nhỏ nhất ở đầu, biên `1`/`100`).

In [14]:
impls = [SolutionBrute(), SolutionRunning(), SolutionAccumulate(), SolutionRecords(), Solution()]
tests = [
    ([5, 3, 4, 1, 3, 2], [5, 3, 3, 1, 1, 1]),
    ([1, 2, 4, 6, 7], [1, 1, 1, 1, 1]),
    ([7], [7]),                                  # n = 1
    ([100], [100]),                              # biên trên
    ([1], [1]),                                  # biên dưới
    ([5, 5, 5, 5], [5, 5, 5, 5]),                # toàn bằng nhau
    ([9, 8, 7, 6], [9, 8, 7, 6]),                # giảm dần: mỗi vị trí là kỷ lục
    ([2, 4, 6, 8], [2, 2, 2, 2]),                # tăng dần
    ([9, 9, 9, 1], [9, 9, 9, 1]),                # nhỏ nhất ở cuối
    ([3, 1, 2, 1, 5], [3, 1, 1, 1, 1]),          # bằng kỷ lục không đổi
    ([100, 1, 100, 1, 100], [100, 1, 1, 1, 1]),  # biên xen kẽ
    ([4, 5, 3, 6, 2, 7, 1], [4, 4, 3, 3, 2, 2, 1]),
]
for cost, exp in tests:
    assert 1 <= len(cost) <= 100 and all(1 <= v <= 100 for v in cost)
    for im in impls:
        got = im.minCosts(list(cost))
        assert got == exp, (type(im).__name__, cost, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 12


Test 2: ngẫu nhiên có seed, đúng constraints (`1 <= n <= 100`, `1 <= cost[i] <= 100`). Oracle độc lập tính bằng `min(cost[:i + 1])` viết thành list comprehension.

In [15]:
import random
random.seed(3502)

def oracle(cost):
    return [min(cost[:i + 1]) for i in range(len(cost))]

for t in range(1500):
    n = random.randint(1, 100)
    hi_val = random.choice([3, 10, 100])
    cost = [random.randint(1, hi_val) for _ in range(n)]
    assert 1 <= n <= 100 and all(1 <= v <= 100 for v in cost)
    exp = oracle(cost)
    for im in impls:
        assert im.minCosts(list(cost)) == exp, (type(im).__name__, cost)
print("1500 test ngẫu nhiên OK")

1500 test ngẫu nhiên OK


Test 3: tính chất — không sửa input; `answer[0] == cost[0]`; `answer` **không tăng**; `answer[i] <= cost[i]`; `answer[-1] == min(cost)`; mỗi `answer[i]` là một phần tử của `cost[0..i]`.

In [16]:
random.seed(502)
for _ in range(500):
    n = random.randint(1, 100)
    cost = [random.randint(1, 100) for _ in range(n)]
    snap = list(cost)
    ans = Solution().minCosts(cost)
    assert cost == snap and len(ans) == n
    assert ans[0] == cost[0]
    assert all(ans[i] >= ans[i + 1] for i in range(n - 1))
    assert all(ans[i] <= cost[i] for i in range(n))
    assert ans[-1] == min(cost)
    assert all(ans[i] in cost[:i + 1] for i in range(n))
print("tính chất (không sửa input, không tăng, <= cost, cuối = min): OK")

tính chất (không sửa input, không tăng, <= cost, cuối = min): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: trả lại chính `cost` (không lấy min)

In [17]:
def wrong_identity(cost):
    return list(cost)                            # SAI: bỏ qua việc "trả cho người rẻ hơn phía trước rồi đổi miễn phí"

cost = [5, 3, 4, 1, 3, 2]
print("sai :", wrong_identity(cost))
print("đúng:", Solution().minCosts(cost))
assert wrong_identity(cost) != Solution().minCosts(cost)

sai : [5, 3, 4, 1, 3, 2]
đúng: [5, 3, 3, 1, 1, 1]


Vì sao sai: tại vị trí `2` có thể trả `3` (người 1) rồi đổi miễn phí với người 2, rẻ hơn `cost[2] = 4`. **Sửa:** `answer[i] = min(cost[0..i])`.

### Lỗi 2: dùng min **toàn cục** cho mọi vị trí

In [18]:
def wrong_global_min(cost):
    m = min(cost)
    return [m] * len(cost)                       # SAI: người rẻ nhất có thể đứng SAU vị trí cần tới

cost = [5, 3, 4, 1, 3, 2]
print("sai :", wrong_global_min(cost))
print("đúng:", Solution().minCosts(cost))
assert wrong_global_min(cost) != Solution().minCosts(cost)

sai : [1, 1, 1, 1, 1, 1]
đúng: [5, 3, 3, 1, 1, 1]


Vì sao sai: để tới vị trí `0` chỉ có thể trả `cost[0] = 5`; người rẻ nhất (`cost[3] = 1`) đứng **sau** vị trí `0` nên bạn không đổi chỗ được với họ để tới `0`. Chỉ các người ở vị trí `<= i` mới giúp được. **Sửa:** min **tiền tố**, không phải min toàn mảng.

### Lỗi 3: dùng min **hậu tố** (suffix) thay vì tiền tố

In [19]:
def wrong_suffix_min(cost):
    n = len(cost)
    ans = [0] * n
    cur = float("inf")
    for i in range(n - 1, -1, -1):               # SAI: duyệt ngược tính min của cost[i..n-1]
        cur = min(cur, cost[i])
        ans[i] = cur
    return ans

cost = [5, 3, 4, 1, 3, 2]
print("sai :", wrong_suffix_min(cost))
print("đúng:", Solution().minCosts(cost))
assert wrong_suffix_min(cost) != Solution().minCosts(cost)

sai : [1, 1, 1, 1, 2, 2]
đúng: [5, 3, 3, 1, 1, 1]


Vì sao sai: hậu tố cho `[1,1,1,1,2,2]` — tại vị trí `0` ra `1` (không thể, phải trả `5`). Hướng đi quan trọng: chỉ người ở **trước hoặc đúng** vị trí `i` giúp tới `i`. **Sửa:** duyệt **xuôi**.

### Lỗi 4: khởi tạo `cur = 0`

In [20]:
def wrong_init_zero(cost):
    cur = 0                                      # SAI: min(0, x) = 0 với mọi x >= 1
    ans = []
    for c in cost:
        cur = min(cur, c)
        ans.append(cur)
    return ans

cost = [5, 3, 4, 1, 3, 2]
print("sai :", wrong_init_zero(cost))
print("đúng:", Solution().minCosts(cost))
assert wrong_init_zero(cost) != Solution().minCosts(cost)

sai : [0, 0, 0, 0, 0, 0]
đúng: [5, 3, 3, 1, 1, 1]


Vì sao sai: `cost[i] >= 1` nên `min(0, cost[i]) = 0` — `cur` kẹt ở `0` mãi, mảng trả về toàn số `0`. **Sửa:** `cur = cost[0]` (hoặc `float('inf')`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chi phí nhỏ nhất để tới từng vị trí, quay lui miễn phí" -> thường quy về **min tiền tố**.
- Đọc kỹ **chiều** của luật (ai đổi chỗ miễn phí, ai phải trả) để biết là **tiền tố** hay **hậu tố**.
- Công thức vàng: `best[i] = min(best[i - 1], a[i])` — một lượt, một biến.
- `accumulate(a, min)` (hoặc `max`) là cách viết một dòng cho min/max tiền tố.
- Khởi tạo min bằng **phần tử đầu** hoặc `inf`, không bao giờ `0`.
- Kiểm tra: `n = 1`, dãy giảm dần (mỗi vị trí là kỷ lục), dãy tăng dần (một bậc duy nhất).

## 15. Biến thể của bài

1. **Max tiền tố** (`accumulate(a, max)`): "điểm cao nhất tính đến nay".
2. **Min hậu tố**: chiều ngược lại — chi phí khi bạn đứng ở **đầu** hàng.
3. **Đáp án là vị trí (chỉ số)** của người rẻ nhất cho từng `i` thay vì giá trị.
4. Cả hai chiều: `answer[i] = min(min tiền tố, min hậu tố)` (hai pass).
5. **Có chi phí đi bộ**: tới `i` còn phải trả thêm `i - j` cho mỗi bước (khi đó cần quy hoạch động thay vì min thuần).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute `min(cost[:i+1])` | tính lại min từ đầu cho mỗi `i` | `O(n^2)` | `O(n)` | hiểu đề, oracle |
| 2. Một biến `cur` | min đang chạy | `O(n)` | `O(1)` thêm | **chuẩn, nên nộp** |
| 3. `accumulate(cost, min)` | thư viện chuẩn | `O(n)` | `O(n)` | một dòng, Pythonic |
| 4. Kỷ lục + điền đoạn | chỉ ghi nơi min đổi | `O(n)` | `O(n)` | thấy cấu trúc bậc thang |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minCosts([8, 6, 7, 2, 9, 1])` bằng bao nhiêu?

**Bài 2 (tính tay):** `cost = [4, 4, 2, 5, 2, 3]`: lập bảng `cur` và `answer`; những chỉ số nào là kỷ lục mới (nhỏ hơn **thật sự** mọi giá trị trước)?

**Bài 3 (điền TODO):** hoàn thành `prefix_min(a)` trả về mảng min tiền tố.

**Bài 4 (tự viết lại):** viết `prefix_argmin(cost)` trả về **chỉ số** (nhỏ nhất, ưu tiên chỉ số đầu nếu bằng nhau) của người rẻ nhất trong `cost[0..i]` cho từng `i`.

**Bài 5 (mini challenge):** viết `suffix_min(a)` trả về min hậu tố và `two_sided_min(a)` với `two_sided[i] = min(prefix[i], suffix[i])`; kiểm tra bằng `assert`.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minCosts([8, 6, 7, 2, 9, 1]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def prefix_min(a):
    # TODO: cur = a[0]; duyệt a, cập nhật cur = min(cur, x) rồi append
    return []

[8, 6, 6, 2, 2, 1]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** đáp án tại `i` là giá trị nhỏ nhất trong các phần tử từ đầu tới `i`.

**Mức 2 (vừa):** Bài 3: giống Cách 2. Bài 4: nhớ `best_idx`; chỉ đổi khi `cost[i] < cost[best_idx]` (dấu `<` thật sự để giữ chỉ số đầu khi bằng nhau). Bài 5: `suffix_min` duyệt từ cuối về đầu, rồi `[min(p, s) for p, s in zip(...)]`.

**Mức 3 (gần đáp án):** Bài 1: `[8, 6, 6, 2, 2, 1]`. Bài 2: `[4, 4, 2, 2, 2, 2]`; kỷ lục mới ở chỉ số `0` (khởi đầu) và `2` (`2 < 4`).

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
assert Solution().minCosts([8, 6, 7, 2, 9, 1]) == [8, 6, 6, 2, 2, 1]

# Bài 2
assert Solution().minCosts([4, 4, 2, 5, 2, 3]) == [4, 4, 2, 2, 2, 2]

# Bài 3
def prefix_min(a):
    cur = a[0]
    out = []
    for x in a:
        if x < cur:
            cur = x
        out.append(cur)
    return out
assert prefix_min([5, 3, 4, 1, 3, 2]) == [5, 3, 3, 1, 1, 1]
assert prefix_min([7]) == [7]
assert prefix_min([1, 2, 3]) == [1, 1, 1]

# Bài 4
def prefix_argmin(cost):
    best_idx = 0
    out = []
    for i, c in enumerate(cost):
        if c < cost[best_idx]:
            best_idx = i
        out.append(best_idx)
    return out
assert prefix_argmin([5, 3, 4, 1, 3, 2]) == [0, 1, 1, 3, 3, 3]
assert prefix_argmin([4, 4, 2, 5, 2, 3]) == [0, 0, 2, 2, 2, 2]
assert [cost_i for cost_i in [5, 3, 4, 1, 3, 2]] and \
    [[5, 3, 4, 1, 3, 2][j] for j in prefix_argmin([5, 3, 4, 1, 3, 2])] == Solution().minCosts([5, 3, 4, 1, 3, 2])

# Bài 5
def suffix_min(a):
    out = [0] * len(a)
    cur = a[-1]
    for i in range(len(a) - 1, -1, -1):
        cur = min(cur, a[i])
        out[i] = cur
    return out
def two_sided_min(a):
    p, s = prefix_min(a), suffix_min(a)
    return [min(x, y) for x, y in zip(p, s)]
assert suffix_min([5, 3, 4, 1, 3, 2]) == [1, 1, 1, 1, 2, 2]
assert two_sided_min([5, 3, 4, 1, 3, 2]) == [1, 1, 1, 1, 1, 1]
assert two_sided_min([2, 9, 9]) == [2, 2, 2]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "min tiền tố":**

```python
cur = cost[0]
ans = []
for c in cost:
    if c < cur:
        cur = c
    ans.append(cur)
return ans
# hoặc: return list(accumulate(cost, min))
```

**Checklist trước khi submit:**

- [ ] Đáp án tại `i` là min của `cost[0..i]` (tiền tố), không phải toàn mảng, không phải hậu tố?
- [ ] `cur` khởi tạo bằng `cost[0]` hoặc `inf` (không phải `0`)?
- [ ] Mảng trả về có độ dài `n` và `answer[0] == cost[0]`?
- [ ] Không sửa mảng `cost` đầu vào?
- [ ] Đã thử `n = 1`, dãy giảm dần, dãy tăng dần?
- [ ] Chữ ký đúng `minCosts(self, cost: List[int]) -> List[int]`?